# NoC Architecture Generation - Training on Google Colab

This notebook trains a Mistral-7B model with QLoRA for Network-on-Chip architecture generation.

**Setup Instructions:**
1. Upload this notebook to Google Colab
2. Enable GPU: Runtime → Change runtime type → T4 GPU
3. Upload your data files or mount Google Drive
4. Run all cells

## 1. Setup Environment

In [ ]:
import torch
import subprocess

# Check if CUDA is available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Verify GPU details
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
else:
    print("GPU not available. Make sure to enable GPU in Runtime > Change runtime type > GPU")

In [ ]:
!pip install -q transformers==4.41.2 trl==0.8.6 accelerate==0.30.1 datasets==2.18.0 peft==0.11.1 bitsandbytes

In [ ]:
# !pip install -q transformers==4.56.2
# !pip install -q trl==0.8.6
# !pip install -q accelerate==0.30.1
# !pip install -q datasets==2.18.0
# !pip install -q peft==0.11.1
# !pip install -q bitsandbytes==0.43.1
!pip install -q sentencepiece==0.1.99
!pip install -q protobuf==4.25.3
!pip install -q wandb==0.16.3

## 2. Upload Project Files

**Option A: Upload files manually**
- Click the folder icon on the left
- Upload: `src/`, `data/processed_str/`, `configs/`

**Option B: Clone from GitHub (if you have a repo)**

In [ ]:
# Option B: Clone from GitHub
# !git clone https://github.com/your-username/your-repo.git
# %cd your-repo

## 3. Create Training Script

If you uploaded files, skip this. Otherwise, create the training script here:

In [ ]:
# Create directories
!mkdir -p src data/processed_str configs outputs

In [ ]:
%%writefile src/train_sft.py
import json
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

def make_text(example):
    return {"text": example["prompt"] + example["label"]}

def main():
    # Configuration
    model_name = "mistralai/Mistral-7B-Instruct-v0.2"
    output_dir = "outputs/mistral7b-noc-switch-qlora"
    
    # QLoRA configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    # LoRA configuration
    peft_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    
    # Load tokenizer and model
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, peft_config)
    
    # Load datasets
    print("Loading datasets...")
    train_data = []
    with open("data/processed_str/train.jsonl", "r") as f:
        for line in f:
            train_data.append(json.loads(line))
    
    valid_data = []
    with open("data/processed_str/valid.jsonl", "r") as f:
        for line in f:
            valid_data.append(json.loads(line))
    
    train_dataset = Dataset.from_list(train_data).map(make_text, remove_columns=["prompt", "label"])
    valid_dataset = Dataset.from_list(valid_data).map(make_text, remove_columns=["prompt", "label"])
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Valid samples: {len(valid_dataset)}")
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        evaluation_strategy="epoch",
        fp16=True,
        optim="paged_adamw_8bit",
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        report_to="none",
    )
    
    # Create trainer
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        tokenizer=tokenizer,
        max_seq_length=1024,
        dataset_text_field="text",
    )
    
    # Train
    print("Starting training...")
    trainer.train()
    
    # Save model
    print("Saving model...")
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    print(f"Model saved to {output_dir}")

if __name__ == "__main__":
    main()

## 4. Upload Training Data

Upload your `train.jsonl` and `valid.jsonl` files to the `data/processed_str/` directory using the file browser on the left.

In [ ]:
# Verify data files exist
!ls -lh data/processed_str/

## 5. Start Training

In [ ]:
# Run training
!python src/train_sft.py

## 6. Download Trained Model

After training completes, download your model:

In [ ]:
# Create a zip file of the trained model
!zip -r mistral7b-noc-switch-qlora.zip outputs/mistral7b-noc-switch-qlora/

In [ ]:
# Download the zip file
from google.colab import files
files.download('mistral7b-noc-switch-qlora.zip')

## 7. Test Inference (Optional)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Load base model and adapter
base_model_name = "mistralai/Mistral-7B-Instruct-v0.2"
adapter_path = "outputs/mistral7b-noc-switch-qlora"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(model, adapter_path)

# Test prompt
test_prompt = """Generate a Network-on-Chip architecture with the following requirements:

Number of nodes: 4
Topology: mesh
Routing algorithm: xy
Buffer size: 4
Link bandwidth: 32

Generate the switch configuration and routing paths:"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)